In [10]:
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

In [11]:
ctype_abund_df = pd.read_csv("data/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_Claude_marker_genes_PC1_ctype_abundance.csv", index_col=0)

In [12]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/gencode.vM35.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs = gtf_subset["attribute"].apply(extract_attributes)
attrs_df = attrs.apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]

# PSI

In [13]:
psi = pd.read_csv(f"data/hahn_2023_cortex_STAR_exon_PSI.csv", index_col=0)
psi.columns = psi.columns.str.replace("-", ".")

common = psi.columns.intersection(ctype_abund_df.index)
psi = psi[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [14]:
# Get PSI data ready to merge on gene IDs
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['exon_id'] = psi.index.values

psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("exon_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

In [7]:
# Correlate each cell type with all PSI events across samples

psi_numeric = psi_anno.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

corr_results = {}
for ct in ctype_abund_df.columns:
    corr_results[ct] = psi_numeric.T.corrwith(ctype_abund_df[ct])

psi_corr_df = pd.DataFrame(corr_results)
psi_corr_df.insert(0, 'Gene', psi_anno['gene_name'])

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/

In [9]:
psi_corr_df.to_csv(f"data/corrs/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_Claude_marker_genes_PC1_ctype_abundance_exon_PSI_corr.csv")

# Counts

In [14]:
# Do the same analysis for exon counts to have as reference

In [10]:
counts = pd.read_csv(f"data/hahn_2023_cortex_STAR_exon_counts.csv", index_col=0)
counts.columns = counts.columns.str.replace("-", ".")

common = counts.columns.intersection(ctype_abund_df.index)
counts = counts[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [ ]:
# Get data ready to merge on gene IDs

counts['gene_id'] = counts.index.str.split("_").str[0]
counts['exon_id'] = counts.index.values

counts_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], counts, on="gene_id", how="right")
counts_anno = counts_anno.set_index("exon_id").rename_axis(None)
counts_anno = counts_anno.drop(columns=["gene_id"])

In [13]:
# Correlate each cell type with all PSI events across samples

counts_numeric = counts_anno.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

corr_results = {}
for ct in ctype_abund_df.columns:
    corr_results[ct] = counts_numeric.T.corrwith(ctype_abund_df[ct])

corr_df = pd.DataFrame(corr_results)
corr_df.insert(0, 'Gene', counts_anno['gene_name'])

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/

In [35]:
corr_df.sort_values("Vip Sncg", ascending=False).head(30)

,Gene,All Neuronal,All Glutamatergic,All GABAergic,CGE Class,MGE Class,L4 IT,L6 CT,L6 IT Car3,Sncg,Pvalb,Sst,Sst Chodl,Astro,Oligo,Endo,VLMC,Peri,L4 L5 IT,Vip Sncg
ENSMUSG00000055805_ProteinCoding_3,Fmnl1,0.753897,0.791032,0.648797,0.426022,0.564289,0.891417,0.640179,0.431702,0.441551,0.711935,0.325678,-0.076222,0.731615,0.547784,0.731924,-0.153727,-0.175505,0.713848,0.929322
ENSMUSG00000038530_ProteinCoding_1,Rgs4,0.802327,0.824725,0.705824,0.471797,0.615785,0.912387,0.625273,0.487727,0.505491,0.741600,0.387070,-0.020654,0.784123,0.532315,0.775722,-0.102908,-0.119887,0.738956,0.907195
ENSMUSG00000019960_ProteinCoding_1,Dusp6,0.753487,0.781302,0.631625,0.395528,0.511520,0.866163,0.622218,0.371830,0.419718,0.770115,0.263514,-0.142327,0.712219,0.570148,0.688738,-0.205012,-0.237685,0.755736,0.897446
ENSMUSG00000049225_ProteinCoding_1,Pdp1,0.747773,0.782298,0.602741,0.359113,0.502795,0.845881,0.644522,0.352057,0.399568,0.743244,0.245790,-0.153640,0.697630,0.566333,0.693690,-0.228635,-0.247241,0.732753,0.894748
ENSMUSG00000049225_ProteinCoding_2,Pdp1,0.762943,0.791024,0.627637,0.387056,0.532156,0.858532,0.645824,0.380028,0.430027,0.753611,0.278773,-0.119774,0.721477,0.564496,0.712777,-0.194658,-0.212002,0.740185,0.891273
ENSMUSG00000048978_other_1,Nrsn1,0.778445,0.800750,0.751152,0.592675,0.697615,0.910761,0.560775,0.607546,0.567435,0.617346,0.509455,0.139252,0.805633,0.467955,0.790694,0.043257,0.048607,0.666206,0.887137
ENSMUSG00000019990_ProteinCoding_1,Pde7b,0.722770,0.750547,0.648782,0.448489,0.574273,0.895174,0.510036,0.459527,0.443722,0.657475,0.361744,-0.027911,0.723744,0.439102,0.780089,-0.101892,-0.108432,0.675325,0.875097
ENSMUSG00000038244_NMD_1,Mical2,0.846698,0.867501,0.759956,0.551586,0.672353,0.822378,0.710480,0.522638,0.586533,0.735501,0.437911,0.072253,0.814146,0.636637,0.775781,-0.066492,-0.067298,0.774671,0.872293
ENSMUSG00000021301_ProteinCoding_1,Hecw1,0.703269,0.740168,0.613661,0.407048,0.528754,0.864713,0.584012,0.407842,0.400640,0.669517,0.304573,-0.088043,0.681821,0.489814,0.676674,-0.168975,-0.177067,0.679865,0.865734
ENSMUSG00000029705_ProteinCoding_2,Cux1,0.760068,0.783855,0.716892,0.529050,0.637056,0.907874,0.551285,0.527509,0.510183,0.697992,0.420685,0.041493,0.774940,0.471320,0.778715,-0.045664,-0.061562,0.708908,0.864238


In [33]:
corr_df.to_csv(f"data/corrs/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_Claude_marker_genes_PC1_ctype_abundance_exon_counts_corr.csv")

# Gene expr

In [16]:
bulk_expr = pd.read_csv("hahn_2023_cortex_STAR_counts_TMMF_SampleNetworks/All_03-44-55/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed.csv")
bulk_expr.columns.values[0] = "Gene"

In [17]:
mean_expr = pd.DataFrame({
    'Index': range(len(bulk_expr)),
    'Gene': bulk_expr.iloc[:, 0],
    'Expr': bulk_expr.iloc[:, 1:].mean(axis=1)
})

keep = mean_expr.loc[mean_expr.groupby('Gene')['Expr'].idxmax(), 'Index']
bulk_expr_filtered = bulk_expr.iloc[keep.values]

In [18]:
bulk_expr_filtered = bulk_expr_filtered.set_index("Gene")
common = bulk_expr_filtered.columns.intersection(ctype_abund_df.index)
bulk_expr_filtered = bulk_expr_filtered[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [19]:
# Correlate each cell type with all PSI events across samples

# expr_numeric = bulk_expr_filtered.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

expr_corr_results = {}
for ct in ctype_abund_df.columns:
    expr_corr_results[ct] = bulk_expr_filtered.T.corrwith(ctype_abund_df[ct])

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/

In [20]:
expr_corr_df = pd.DataFrame(expr_corr_results)
expr_corr_df.head()

,All Neuronal,All Glutamatergic,All GABAergic,CGE Class,MGE Class,L4 IT,L6 CT,L6 IT Car3,Sncg,Pvalb,Sst,Sst Chodl,Astro,Oligo,Endo,VLMC,Peri,L4 L5 IT,Vip Sncg
Gene,,,,,,,,,,,,,,,,,,,
0610005C13Rik,0.556690,0.526353,0.628878,0.564010,0.626058,0.387537,0.338188,0.635236,0.633171,0.278546,0.609607,0.420586,0.630310,0.258084,0.574153,0.386862,0.357311,0.393277,0.383543
0610006L08Rik,0.120974,0.116904,0.158819,0.226994,0.189076,0.256678,0.010762,0.202984,0.131719,0.017486,0.215986,0.153547,0.196186,0.038707,0.199845,0.093098,0.151248,0.034001,0.232925
0610009E02Rik,0.811142,0.782965,0.873884,0.823479,0.888328,0.532076,0.525484,0.873987,0.849673,0.479225,0.830166,0.559919,0.882111,0.490804,0.792256,0.452640,0.489555,0.626618,0.472594
0610009L18Rik,0.744838,0.700963,0.872749,0.895119,0.932989,0.349264,0.447919,0.955566,0.923486,0.331053,0.946740,0.740734,0.851526,0.422283,0.730444,0.663865,0.696194,0.513833,0.273351
0610010K14Rik,0.732209,0.712172,0.757924,0.655349,0.740337,0.573436,0.523384,0.726905,0.737491,0.472566,0.682825,0.375037,0.785111,0.457620,0.761761,0.389923,0.341722,0.548460,0.516871


In [21]:
expr_corr_df.to_csv(f"data/corrs/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_Claude_marker_genes_PC1_ctype_abundance_gene_expr_corr.csv")